# 5. MLP units as logic

**The question:** attention brought the bytes. How does the MLP turn them into the output's bits?

Each unit of the MLP is a ReLU acting as an AND gate.

In [1]:
import numpy as np

import helpers
from helpers import forward, load, named, rows, stream

model = load('base64')
r = model.r
mlp = r.mlp
tokens = stream(model)
print('units:', len(mlp.units))
for u in mlp.units:
  if u.startswith('encode place 0'):
    print(' ', u)

units: 54
  encode place 0: first.bit2 -> out bit 0
  encode place 0: first.bit3 -> out bit 1
  encode place 0: first.bit4 -> out bit 2
  encode place 0: first.bit5 -> out bit 3
  encode place 0: first.bit6 -> out bit 4
  encode place 0: first.bit7 -> out bit 5
  encode place 0: no more groups -> END


## 1. One unit

`encode place 0: first.bit6 -> out bit 4` means: for the first output of a group, copy bit 6 of the first byte to
output bit 4. Its weights:

In [2]:
names = r.S.names()
u = mlp.units.index('encode place 0: first.bit6 -> out bit 4')
reads = {names[i]: float(mlp.W_in[i, u]) for i in np.flatnonzero(mlp.W_in[:, u])}
writes = [names[i] for i in np.flatnonzero(mlp.W_out[u])]
print('reads: ', reads)
print('bias:  ', mlp.b_in[u])
print('writes:', writes)

reads:  {'phase[3]': 1.0, 'mode_enc': 1.0, 'enc_first.bits[6]': 1.0}
bias:   -2.0
writes: ['out_bits[4]']


`phase[3]` is the token that predicts the first output. The unit is ReLU(a + b + c - 2). With inputs of 0 or 1 that
is 1 only when all three are 1:

In [3]:
for a in (0, 1):
  for b in (0, 1):
    for c in (0, 1):
      print(a, b, c, '->', max(0, a + b + c - 2))

0 0 0 -> 0
0 0 1 -> 0
0 1 0 -> 0
0 1 1 -> 0
1 0 0 -> 0
1 0 1 -> 0
1 1 0 -> 0
1 1 1 -> 1


## 2. How the builder finds the wires

The builder never reads `encode_char`'s code. It calls it with one input bit set at a time and sees where the bit lands.

In [4]:
from qkvc.build import wiring
from tasks.rearrange.base64 import encode_char

print('first output is first >> 2:')
for which, k, o in wiring(encode_char, 0, [8, 8]):
  print(f'  {("first", "second")[which]} bit {k} -> out bit {o}')
print('combine with only bit 6 of first set:', encode_char(1 << 6, 0, 0), '= bit 4')

first output is first >> 2:
  first bit 2 -> out bit 0
  first bit 3 -> out bit 1
  first bit 4 -> out bit 2
  first bit 5 -> out bit 3
  first bit 6 -> out bit 4
  first bit 7 -> out bit 5
combine with only bit 6 of first set: 16 = bit 4


## 3. Which units fire for `S`

The vector from chapter 4 goes through the MLP. `H` has bits 3 and 6 on, so two units fire.

In [5]:
X = rows(model, tokens[:4])
before, hidden, after = forward(model, X)
for unit, value in zip(mlp.units, hidden):
  if value > 0.5:
    print(f'{value:.0f}  {unit}')

1  encode place 0: first.bit3 -> out bit 1
1  encode place 0: first.bit6 -> out bit 4


In [6]:
print({k: v for k, v in named(model, after).items() if k.startswith('out')})

{'out_bits[1]': 1.0, 'out_bits[4]': 1.0}


Output bits 1 and 4 are `010010`, which is 18: the character `S`.

## 4. All four outputs of the first group

In [7]:
for n, step in enumerate(model.steps('encode', helpers.TEXT, detail=True)):
  if n == 4:
    break
  print(f"output {n + 1}: {step['name']}")
  for unit in step['units']:
    print('   ', unit)

output 1: S
    encode place 0: first.bit3 -> out bit 1
    encode place 0: first.bit6 -> out bit 4
output 2: G
    encode place 1: second.bit5 -> out bit 1
    encode place 1: second.bit6 -> out bit 2
output 3: V
    encode place 2: first.bit0 -> out bit 2
    encode place 2: first.bit2 -> out bit 4
    encode place 2: second.bit6 -> out bit 0
output 4: s
    encode place 3: first.bit2 -> out bit 2
    encode place 3: first.bit3 -> out bit 3
    encode place 3: first.bit5 -> out bit 5


Encoding uses 24 wires: 6 bits for each of the 4 places. No unit computes anything; each one copies a bit to its place.
Base64 is pure wiring. Chapter 9 shows algorithms that need more.